### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [35]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


d:\ESTUDIOS\posgrado\Materias\CEIA (4) Procesamiento de lenguaje natural I\procesamiento_lenguaje_natural\.venv\Scripts\python.exe: No module named pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [36]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # CountVectorizer: convierte una colección de documentos de texto en una matriz de tokens contados. TfidfVectorizer: convierte una colección de documentos de texto en una matriz de características TF-IDF.
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB # MultinomialNB: clasificador Naive Bayes para datos discretos, como el conteo de palabras. ComplementNB: una variante de MultinomialNB que es más adecuada para clases desbalanceadas.
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [37]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [38]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [39]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [40]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [41]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

KeyboardInterrupt: 

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
for t in y_train[:10]:
    print(t, '=', newsgroups_train.target_names[t])

7 = rec.autos
4 = comp.sys.mac.hardware
4 = comp.sys.mac.hardware
1 = comp.graphics
14 = sci.space
16 = talk.politics.guns
13 = sci.med
3 = comp.sys.ibm.pc.hardware
2 = comp.os.ms-windows.misc
4 = comp.sys.mac.hardware


Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1. Vectorizar documentos

In [ ]:
rng = np.random.default_rng(seed=42) # Fijar la semilla
random_idxs = rng.choice(X_train.shape[0], size=5, replace=False) # Se eligen 5 índices aleatorios de documentos del conjunto de entrenamiento sin reemplazo
print(f'Documentos seleccionados: {random_idxs}')

Documentos seleccionados: [8754 4965 7404 1009 4899]


In [ ]:
# Funcion para generar un resumen del texto: solo para debugar
def resumen(texto, n=200):
    txt = texto.strip().replace('\n', ' ') # Se eliminan los saltos de linea y se reemplazan por espacios
    if len(txt):
        return txt[:n] # Se devuelve un resumen de los primeros n caracteres del texto
    else:
        return txt # Se devuelve el texto completo si es mas corto que n caracteres

for idx in random_idxs:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    top5 = np.argsort(cossim)[::-1][1:6] # Se excluye el propio documento
    
    clase_base = newsgroups_train.target_names[y_train[idx]] # Se obtiene la clase del documento base

    print(f'Documento base idx={idx} | clase: {clase_base}')
    print(f'Texto: {resumen(newsgroups_train.data[idx])}')
    print('=' * 90)
    
    for rank, i in enumerate(top5, start=1):
        clase_i = newsgroups_train.target_names[y_train[i]] # Clase del documento similar
        match = 'MATCH' if clase_base == clase_i else 'DIFF'
        print(f'  {rank}. idx={i:5d} | sim={cossim[i]:.3f} | [{match}] clase: {clase_i}')
        print(f'     {resumen(newsgroups_train.data[i], 160)}')
    print()

Documento base idx=8754 | clase: talk.religion.misc
Texto: /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your bus
  1. idx= 6552 | sim=0.490 | [MATCH] clase: talk.religion.misc
     If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, then I do so. There have been bad habits 
  2. idx=10613 | sim=0.481 | [MATCH] clase: talk.religion.misc
     /(hudson) /Yes you do.  Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way.  Maybe unpleasant, but that doesn't /
  3. idx= 3616 | sim=0.465 | [MATCH] clase: talk.religion.misc
     And I maintain:  Some people do not want to enter into the light and the knowledge that they alone are their own masters, because they fear it; they are too afr
  4. idx= 8726 | sim=0.460 | [DIFF] clase: talk.polit

Conclusiones

- Cuando el documento base tiene vocabulario técnico distintivo (idx=8754, 4965), TF-IDF acierta: mayoría de MATCH y similaridades altas (0.35–0.49).
- Los `DIFF` casi siempre caen en clases parecidas (`comp.sys.mac` y `comp.sys.ibm`, `talk.religion` y `talk.politics`), lo que muestra que la representación captura afinidad temática aunque no siempre coincida con la etiqueta exacta.
- Falla con documentos muy cortos y genéricos (idx=7404: 0 MATCH), y también cuando el contenido real no coincide con la etiqueta (idx=4899 está en `sci.crypt` pero habla de política, y sus vecinos lo reflejan).
- En resumen: TF-IDF y coseno mide solapamiento de vocabulario ponderado.

### 2. Construir un modelo de clasificación por prototipos (tipo zero-shot)

In [ ]:
# X_test ya vectorizado con el TfidfVectorizer ajustado en train
sim_matrix = cosine_similarity(X_test, X_train) # Matriz de similaridad entre cada documento del test con el train

# Para cada doc de test, índice del doc de train más similar
nearest_train_idx = np.argmax(sim_matrix, axis=1)

# La clase predicha es la del vecino más cercano en train
y_pred_proto = y_train[nearest_train_idx]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-macro (1-NN por similaridad coseno sobre TF-IDF): {f1_proto:.4f}')

F1-macro (1-NN por similaridad coseno sobre TF-IDF): 0.5050


Conclusiones

- F1-macro = 0.5050 esta por debajo del 0.5854 del Naive Bayes Multinomial base sobre la misma vectorización.
- 1-NN es frágil: la predicción depende de un único documento de train, que puede ser corto, ruidoso o mal etiquetado. Naive Bayes en cambio agrega evidencia de todo el vocabulario de cada clase, lo que lo hace más robusto.
- Aun así, un 50% de F1-macro sobre 20 clases muestra que la sola señal de "vocabulario compartido" ya lleva bastante lejos, sin ningún entrenamiento explícito.

### 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

Se prueban combinaciones de:
- Vectorizador: `CountVectorizer` y `TfidfVectorizer` (sin tocar `ngram_range`).
- Hiperparámetros del vectorizador: `min_df`, `max_df`, `stop_words`, `sublinear_tf`.
- Modelo: `MultinomialNB` y `ComplementNB` con distintos valores de `alpha`.

In [ ]:
from itertools import product

vect_configs = [] # Configuraciones a probar

# CountVectorizer
for min_df, max_df, sw in product([1, 2, 5], [1.0, 0.95], [None, 'english']):
    vect_configs.append(('count', {'min_df': min_df, 'max_df': max_df, 'stop_words': sw}))

# TfidfVectorizer (incluye sublinear_tf)
for min_df, max_df, sw, sub in product([1, 2, 5], [1.0, 0.95], [None, 'english'], [False, True]):
    vect_configs.append(('tfidf', {'min_df': min_df, 'max_df': max_df, 'stop_words': sw, 'sublinear_tf': sub}))


model_configs = [] # Modelos a probar

# Alpha para Naive Bayes (suavizado)
for alpha in [0.01, 0.05, 0.1, 0.3, 1.0]:
    model_configs.append(('MultinomialNB', {'alpha': alpha}))
    model_configs.append(('ComplementNB', {'alpha': alpha}))

results = [] # Resultados de las combinaciones

for vect_name, vect_params in vect_configs:
    if vect_name == 'count':
        vect = CountVectorizer(**vect_params)
    else:
        vect = TfidfVectorizer(**vect_params)

    Xtrain = vect.fit_transform(newsgroups_train.data)
    Xtest = vect.transform(newsgroups_test.data)

    for model_name, model_params in model_configs:
        
        clf = MultinomialNB(**model_params) if model_name == 'MultinomialNB' else ComplementNB(**model_params)
        
        clf.fit(Xtrain, y_train)
        
        y_hat = clf.predict(Xtest)
        f1 = f1_score(y_test, y_hat, average='macro')
        
        results.append({
            'vect': vect_name, 'vect_params': vect_params,
            'model': model_name, 'model_params': model_params,
            'f1_macro': f1,
        })

results.sort(key=lambda r: r['f1_macro'], reverse=True)

print(f'Total combinaciones probadas: {len(results)}\n')
print('Top 10:')
for r in results[:10]:
    print(f"  F1={r['f1_macro']:.4f} | {r['vect']:5s} {r['vect_params']} | {r['model']} {r['model_params']}")

print('\nPeor 3:')
for r in results[-3:]:
    print(f"  F1={r['f1_macro']:.4f} | {r['vect']:5s} {r['vect_params']} | {r['model']} {r['model_params']}")

Total combinaciones probadas: 360

Top 10:
  F1=0.6999 | tfidf {'min_df': 1, 'max_df': 1.0, 'stop_words': None, 'sublinear_tf': False} | ComplementNB {'alpha': 0.3}
  F1=0.6999 | tfidf {'min_df': 1, 'max_df': 0.95, 'stop_words': None, 'sublinear_tf': False} | ComplementNB {'alpha': 0.3}
  F1=0.6993 | tfidf {'min_df': 1, 'max_df': 1.0, 'stop_words': 'english', 'sublinear_tf': False} | ComplementNB {'alpha': 0.3}
  F1=0.6993 | tfidf {'min_df': 1, 'max_df': 0.95, 'stop_words': 'english', 'sublinear_tf': False} | ComplementNB {'alpha': 0.3}
  F1=0.6990 | tfidf {'min_df': 1, 'max_df': 1.0, 'stop_words': None, 'sublinear_tf': True} | ComplementNB {'alpha': 0.3}
  F1=0.6990 | tfidf {'min_df': 1, 'max_df': 0.95, 'stop_words': None, 'sublinear_tf': True} | ComplementNB {'alpha': 0.3}
  F1=0.6979 | tfidf {'min_df': 1, 'max_df': 1.0, 'stop_words': 'english', 'sublinear_tf': True} | ComplementNB {'alpha': 0.3}
  F1=0.6979 | tfidf {'min_df': 1, 'max_df': 0.95, 'stop_words': 'english', 'sublinear_tf

Conclusiones

- Mejor F1-macro = 0.6999 con `TfidfVectorizer` + `ComplementNB(alpha=0.3)`, frente al 0.5854 del baseline (`MultinomialNB(alpha=1.0)` sobre TF-IDF default). Se observa una mejora.
- ComplementNB es más robusto ante el desbalance moderado entre las 20 clases.
- TF-IDF > Count: los peores 3 son todos `CountVectorizer` con `alpha=1.0`. Ponderar por IDF y ajustar el suavizado importa más que filtrar stopwords o cortar vocabulario.
- Los parámetros del vectorizador (`min_df`, `max_df`, `stop_words`, `sublinear_tf`) tienen efecto marginal una vez elegidos TF-IDF + ComplementNB + alpha bajo — los primeros 8 resultados difieren muy poco.

### 4. Transponer la matriz documento-término.

In [ ]:
X_train_T = X_train.T  # shape: (vocab_size, n_docs)
print(f'shape termino-documento: {X_train_T.shape}')

# Palabras elegidas
palabras = ['god', 'car', 'encryption', 'nasa', 'baseball']

vocab = tfidfvect.vocabulary_

for palabra in palabras:
    if palabra not in vocab:
        print(f'"{palabra}" no está en el vocabulario\n')
        continue

    word_idx = vocab[palabra]
    word_vec = X_train_T[word_idx]

    # Similaridad coseno de esta palabra contra TODAS las palabras
    sims = cosine_similarity(word_vec, X_train_T)[0]
    top5 = np.argsort(sims)[::-1][1:6]  # excluir la palabra misma

    print(f'Palabra base: "{palabra}"')
    for rank, i in enumerate(top5, start=1):
        print(f'  {rank}. {idx2word[i]:20s} sim={sims[i]:.3f}')
    print()

shape termino-documento: (101631, 11314)
Palabra base: "god"
  1. jesus                sim=0.269
  2. bible                sim=0.262
  3. that                 sim=0.256
  4. existence            sim=0.255
  5. christ               sim=0.251

Palabra base: "car"
  1. cars                 sim=0.180
  2. criterium            sim=0.177
  3. civic                sim=0.175
  4. owner                sim=0.169
  5. dealer               sim=0.168

Palabra base: "encryption"
  1. microcircuit         sim=0.340
  2. scrambles            sim=0.338
  3. accommodates         sim=0.338
  4. nistnews             sim=0.338
  5. 2758                 sim=0.338

Palabra base: "nasa"
  1. gov                  sim=0.425
  2. 44135                sim=0.361
  3. 433                  sim=0.352
  4. lerc                 sim=0.342
  5. larc                 sim=0.336

Palabra base: "baseball"
  1. tommorrow            sim=0.184
  2. football             sim=0.176
  3. penna                sim=0.173
  4. wintry   

Conclusiones

- Las similaridades son bajas en valor absoluto (0.17–0.43) porque cada palabra es un vector muy disperso en 11314 dimensiones — solo aparece en unos pocos documentos.
- Casos claros y semánticamente correctos:
  - `god` → `jesus`, `bible`, `existence`, `christ`: todo el campo religioso.
  - `car` → `cars`, `civic`, `owner`, `dealer`: automóviles y comercio de autos.
  - `baseball` → `football`, `espn`: deportes y medios deportivos.
- Casos ruidosos: `encryption` y `nasa` traen palabras raras (`microcircuit`, `nistnews`, `2758`, `44135`, `lerc`, `larc`). Son términos que aparecen en muy pocos documentos junto a la palabra base, lo que infla artificialmente el coseno (dos vectores muy dispersos con soporte casi idéntico dan similaridad alta aunque el término sea marginal). `gov` como vecino de `nasa` sí tiene sentido (dominios `.gov`, `nasa.gov`).
- Aparecen también palabras funcionales como `that` cerca de `god`, pasa cuando no se filtran stopwords.
- Conclusión: la matriz término-documento captura co-ocurrencia a nivel documento, no significado. Funciona razonablemente para palabras temáticas frecuentes pero para palabras técnicas o poco frecuentes, el ranking es bastante poco coherente.